In [10]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"

streets_path    = DATA_DIR / "streets" / "Street_Centerline.shp"
tocs_path       = BOUNDARIES_DIR / "processed" / "toc_clean.gpkg"
villages_path   = BOUNDARIES_DIR / "processed" / "village_clean.gpkg"

streets_path, tocs_path, villages_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/streets/Street_Centerline.shp'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/toc_clean.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/village_clean.gpkg'))

In [11]:
PROJECT_CRS = "EPSG:2868"

streets = gpd.read_file(streets_path)
tocs = gpd.read_file(tocs_path)
villages = gpd.read_file(villages_path)

In [12]:
if streets.crs != PROJECT_CRS:
    streets = streets.to_crs(PROJECT_CRS)
if tocs.crs != PROJECT_CRS:
    tocs = tocs.to_crs(PROJECT_CRS)
if villages.crs != PROJECT_CRS:
    villages = villages.to_crs(PROJECT_CRS)

In [14]:
print(streets.columns)
print(tocs.columns)
print(villages.columns)

Index(['OBJECTID', 'STREETPRE', 'STREETNAME', 'STREETTYPE', 'STREETSUF',
       'LTFROMADD', 'LTTOADD', 'RTFROMADD', 'RTTOADD', 'STREETCLAS',
       'STREETCROS', 'ANNAME', 'JURISDICTI', 'CREATOR', 'CREATE_DAT', 'EDITOR',
       'EDIT_DATE', 'STATUS', 'SHAPESTLen', 'geometry'],
      dtype='object')
Index(['OBJECTID', 'APPLICABIL', 'geometry'], dtype='object')
Index(['OBJECTID', 'NAME', 'geometry'], dtype='object')


In [15]:
streets.head()

,OBJECTID,STREETPRE,STREETNAME,STREETTYPE,STREETSUF,LTFROMADD,LTTOADD,RTFROMADD,RTTOADD,STREETCLAS,STREETCROS,ANNAME,JURISDICTI,CREATOR,CREATE_DAT,EDITOR,EDIT_DATE,STATUS,SHAPESTLen,geometry
0,1,W,AUGUSTA,AVE,None,2901,2909,2900,2908,LO,None,W AUGUSTA AVE,Phoenix,STRDBA,2020-03-12,STRDBA,2020-03-12,IN-SERVICE,250.899433,"LINESTRING (637666.729 928156.876, 637415.84 9..."
1,2,W,MORTEN,AVE,None,3001,3005,3000,3004,LO,None,W MORTEN AVE,Private,STRDBA,2020-03-12,STRDBA,2020-03-12,IN-SERVICE,62.283000,"LINESTRING (636980.102 926836.91, 636917.819 9..."
2,3,W,MORTEN,AVE,None,3007,3011,3006,3010,LO,None,W MORTEN AVE,Private,STRDBA,2020-03-12,STRDBA,2020-03-12,IN-SERVICE,63.151000,"LINESTRING (636917.819 926836.91, 636854.668 9..."
3,4,N,33RD,AVE,None,8076,8098,8075,8099,LO,None,N 33RD AVE,Phoenix,STRDBA,2020-03-12,STRDBA,2020-03-12,IN-SERVICE,93.495318,"LINESTRING (634972.686 929368.891, 634972.686 ..."
4,5,N,33RD,AVE,None,8030,8076,8031,8075,LO,None,N 33RD AVE,Phoenix,STRDBA,2020-03-12,STRDBA,2020-03-12,IN-SERVICE,194.837003,"LINESTRING (634972.719 929174.054, 634972.695 ..."


In [45]:
streets["JURISDICTI"].value_counts()

JURISDICTI
Phoenix            59855
Private             9255
ADOT                 950
Maricopa County      358
Scottsdale           111
Glendale              92
Paradise Valley       55
Tempe                 19
State                 18
Cave Creek            11
Peoria                 8
Tolleson               8
Avondale               6
Carefree               1
Unknown                1
Name: count, dtype: int64

In [63]:
# since there are private streets, which don't really affect public street miles, filter them out. also there's 1 "unknown" street, so we can filter that out too.

public_jurisdictions = [
    "Phoenix", "ADOT", "Maricopa County", "State",
    "Scottsdale", "Glendale", "Paradise Valley", "Tempe",
    "Cave Creek", "Peoria", "Tolleson", "Avondale", "Carefree"
]

streets_public = streets[streets["JURISDICTI"].isin(public_jurisdictions)]

# obviously, this project is for the City of Phoenix, but the TOC and village boundaries may cross into other jurisdictions, which is why I want to include all public streets. however, it is possible that the city may want to only consider streets under its jurisdiction to measure its own maintenance responsibilities within a TOC. this can be adjusted later if needed.

In [48]:
# reproject to project CRS for accurate length calculations

streets_public = streets_public.to_crs(PROJECT_CRS)

In [49]:
# shapestlen might make sense... but since geometry exists, it's probably more accurate to just compute directly. since we reprojected to 2868, units are in feet already.

streets_public["shapelen_ft"] = streets_public.geometry.length

# since we are looking at trying to do street miles per acre for the TOCs and villages, we can compute that here as well.

streets_public["shapelen_miles"] = streets_public["shapelen_ft"] / 5280

In [50]:
streets_x_toc = gpd.overlay(
    streets_public[["geometry"]],
    tocs[["OBJECTID", "APPLICABIL", "geometry"]],
    how="intersection",
    keep_geom_type=False
)

# Length of intersection slice
streets_x_toc["length_miles"] = streets_x_toc.geometry.length / 5280.0

In [51]:
street_miles_by_toc = (
    streets_x_toc
    .groupby(["OBJECTID", "APPLICABIL"], as_index=False)
    .agg({"length_miles": "sum"})
    .rename(columns={"length_miles": "street_miles_total"})
)

street_miles_by_toc.head(15)

,OBJECTID,APPLICABIL,street_miles_total
0,1,TOD District - Gateway,45.454557
1,2,TOD District - Eastlake Garfield,36.258417
2,3,TOD District - Midtown,34.679791
3,4,TOD District - Uptown,34.140718
4,5,TOD District - Solano,28.136943
5,6,TOD District - 19North,44.821416
6,7,50th Street Station Area,8.700173
7,8,Capitol Extension Area,35.057163
8,9,I-10 West Extension Area,168.525462
9,10,Northwest Extension Area II,32.665924


In [52]:
# now we can merge back to the tocs demographic data and use the streets to get a fuller picture.

tocs_with_dem = BOUNDARIES_DIR / "tocs" / "toc_demographics.gpkg"

tocs_dem = gpd.read_file(tocs_with_dem)

tocs_full = tocs_dem.merge(
    street_miles_by_toc,
    on="APPLICABIL",
    how="left"
)

tocs_full.head(11)

,OBJECTID_x,APPLICABIL,TOC_ID,income_weighted,rent_weighted,hh_total,median_income_approx,median_rent_approx,intersect_acres,ASNQE001,...,ASORE002,pct_drive_alone,pct_transit,pct_bike,pct_walk,pct_wfh,pct_auto,geometry,OBJECTID_y,street_miles_total
0,1,TOD District - Gateway,0,2.099283e+08,5.097215e+06,4404.074570,47666.824758,1157.386086,2418.944362,13919.344291,...,5197.669073,62.949051,6.885219,1.241587,1.696329,6.949127,80.427979,"POLYGON ((663431.678 894322.427, 663434.469 89...",1,45.454557
1,2,TOD District - Eastlake Garfield,1,1.613483e+08,3.709420e+06,3379.818960,47738.728035,1097.520452,1220.462979,7826.045040,...,2880.542278,62.641426,2.418543,0.639859,4.359989,15.797736,74.261881,"POLYGON ((654751.939 894390.665, 654764.736 89...",2,36.258417
2,3,TOD District - Midtown,2,5.228343e+08,1.150085e+07,7079.314543,73853.809662,1624.571152,1283.000217,11830.041147,...,4738.153812,55.627661,6.914095,0.261316,4.261577,21.698811,65.197519,"POLYGON ((654767.886 907514.625, 654767.631 90...",3,34.679791
3,4,TOD District - Uptown,3,4.255024e+08,8.094779e+06,5546.764459,76711.815672,1459.369509,1332.064285,10889.582934,...,4485.287669,66.306648,4.093010,0.605072,1.733495,23.235733,68.846495,"POLYGON ((649448.684 915530.44, 654791.762 915...",4,34.140718
4,5,TOD District - Solano,4,2.998474e+08,7.064961e+06,5518.487681,54335.073430,1280.234957,1100.265433,14440.725434,...,5030.358146,57.961820,5.484560,0.343444,4.943742,16.830438,69.192002,"POLYGON ((646830.036 919545.608, 646829.436 91...",5,28.136943
5,6,TOD District - 19North,5,6.341370e+08,1.334089e+07,10430.136344,60798.535299,1279.071912,1780.947386,22399.496066,...,8323.108262,60.620135,5.314936,0.999783,1.672430,15.809781,74.140993,"POLYGON ((646874.486 919833.609, 646872.259 91...",6,44.821416
6,7,50th Street Station Area,6,7.584102e+07,1.843089e+06,1397.124710,54283.640432,1319.201558,617.105659,3048.586703,...,1382.606314,63.878418,4.357335,0.690553,1.261305,19.402120,73.359713,"POLYGON ((684257.247 888198.254, 684382.127 88...",7,8.700173
7,8,Capitol Extension Area,7,1.196863e+08,2.520710e+06,2063.514028,58001.224814,1221.561930,1116.466203,5339.173857,...,2134.264228,64.798414,4.853094,1.792639,0.008560,13.533112,79.360778,"POLYGON ((641505.991 895598.434, 643697.045 89...",8,35.057163
8,9,I-10 West Extension Area,8,1.271326e+09,2.939138e+07,22828.810637,55689.542302,1287.468572,7735.922063,77864.054330,...,29945.651488,67.878337,3.040533,0.512905,0.903119,6.445746,86.887576,"POLYGON ((602060.385 899622.267, 602090.536 90...",9,168.525462
9,10,Northwest Extension Area II,9,3.291577e+08,7.308436e+06,5457.748156,60310.174629,1339.093742,1740.704483,13958.933685,...,5212.851623,59.873332,4.929296,0.315743,1.704013,8.975765,80.521258,"POLYGON ((633784.013 939340.634, 644284.495 93...",10,32.665924


In [53]:
# Street miles per acre
tocs_full["street_miles_per_acre"] = (
    tocs_full["street_miles_total"] / tocs_full["intersect_acres"]
)

tocs_full[["TOC_ID", "APPLICABIL", "street_miles_total", "intersect_acres", "street_miles_per_acre"]]

,TOC_ID,APPLICABIL,street_miles_total,intersect_acres,street_miles_per_acre
0,0,TOD District - Gateway,45.454557,2418.944362,0.018791
1,1,TOD District - Eastlake Garfield,36.258417,1220.462979,0.029709
2,2,TOD District - Midtown,34.679791,1283.000217,0.027030
3,3,TOD District - Uptown,34.140718,1332.064285,0.025630
4,4,TOD District - Solano,28.136943,1100.265433,0.025573
5,5,TOD District - 19North,44.821416,1780.947386,0.025167
6,6,50th Street Station Area,8.700173,617.105659,0.014098
7,7,Capitol Extension Area,35.057163,1116.466203,0.031400
8,8,I-10 West Extension Area,168.525462,7735.922063,0.021785
9,9,Northwest Extension Area II,32.665924,1740.704483,0.018766


In [54]:
tocs_full_path = BOUNDARIES_DIR / "tocs" / "toc_full_street_miles.gpkg"

tocs_full.to_file(tocs_full_path, driver="GPKG")
print(f"Saved full data with street miles to: {tocs_full_path}")

Saved full data with street miles to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\tocs\toc_full_street_miles.gpkg


In [55]:
# wash, rinse, repeat for villages:

streets_x_village = gpd.overlay(
    streets_public[["geometry"]],
    villages[["OBJECTID", "NAME", "geometry"]],
    how="intersection",
    keep_geom_type=False
)

streets_x_village["length_miles"] = streets_x_village.geometry.length / 5280.0

In [56]:
street_miles_by_village = (
    streets_x_village
    .groupby(["OBJECTID", "NAME"], as_index=False)
    .agg({"length_miles": "sum"})
    .rename(columns={"length_miles": "street_miles_total"})
)

In [57]:
street_miles_by_village.head(15)

,OBJECTID,NAME,street_miles_total
0,16,Ahwatukee Foothills,265.158636
1,17,Laveen,242.088274
2,18,South Mountain,384.956898
3,19,Estrella,348.538691
4,20,Central City,286.476070
5,21,Camelback East,428.399442
6,22,Maryvale,503.372317
7,23,Encanto,184.972848
8,24,Alhambra,346.444084
9,25,North Mountain,495.887913


In [58]:
villages_with_dem = BOUNDARIES_DIR / "villages" / "village_demographics.gpkg"

villages_dem = gpd.read_file(villages_with_dem)

In [59]:
villages_full = villages_dem.merge(
    street_miles_by_village,
    on=["OBJECTID", "NAME"],
    how="left"
)

villages_full.columns

Index(['OBJECTID', 'NAME', 'ASNQE001_w', 'ASS8E002_w', 'ASS8E003_w',
       'intersect_acres', 'pop_per_acre', 'hh_per_acre', 'ASS9E002_w',
       'ASS9E003_w', 'median_income_approx', 'median_rent_approx',
       'pct_drive_alone', 'pct_transit', 'pct_bike', 'pct_walk', 'pct_wfh',
       'pct_auto', 'geometry', 'street_miles_total'],
      dtype='object')

In [60]:
villages_full["street_miles_per_acre"] = (
    villages_full["street_miles_total"] / villages_full["intersect_acres"]
)

In [61]:
villages_full[["NAME", "street_miles_total", "intersect_acres", "street_miles_per_acre"]]

,NAME,street_miles_total,intersect_acres,street_miles_per_acre
0,Ahwatukee Foothills,265.158636,22832.880975,0.011613
1,Laveen,242.088274,19579.686598,0.012364
2,South Mountain,384.956898,25481.856132,0.015107
3,Estrella,348.538691,26503.624912,0.013151
4,Central City,286.476070,13600.592409,0.021063
5,Camelback East,428.399442,23242.629344,0.018432
6,Maryvale,503.372317,20820.532995,0.024177
7,Encanto,184.972848,6706.535758,0.027581
8,Alhambra,346.444084,12315.416799,0.028131
9,North Mountain,495.887913,22212.332100,0.022325


In [62]:
villages_full_path = BOUNDARIES_DIR / "villages" / "village_full_street_miles.gpkg"

villages_full.to_file(villages_full_path, driver="GPKG")
print(f"Saved full data with street miles to: {villages_full_path}")

Saved full data with street miles to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\villages\village_full_street_miles.gpkg
